In [2]:
pip install opencv-python numpy matplotlib scipy scikit-image

   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.2 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.2 MB 670.4 kB/s eta 0:00:12
   -- ------------------------------------- 0.5/8.2 MB 670.4 kB/s eta 0:00:12
   --- ------------------------------------ 0.8/8.2 MB 684.4 kB/s eta 0:00:11
   --- ------------------------------------ 0.8/8.2 MB 684.4 kB/s eta 0:00:11
   ----- ---------------------------------- 1.0/8.2 MB 689.2 kB/s eta 0:00:11
   ------ --------------------------------- 1.3/8.2 MB 713.8 kB/s eta 0:00:10
   ------ --------------------------------- 1.3/8.2 MB 713.8 kB/s eta 0:00:10
   ------- -------------------------------- 1.6/8.2 MB 729.4 kB/s eta 0:00:10
   ------- -------------------------------- 1.6/8.2 MB 729.4 kB/s eta 0:00:10
   -------- ----------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [3]:

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.ndimage import gaussian_filter
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
import warnings
warnings.filterwarnings('ignore')
 

In [ ]:
#  Core Image Processing Library (Reusable)
class ImageUtils:
    @staticmethod
    def to_float(img: np.ndarray) -> np.ndarray:
        return img.astype(np.float32) / 255.0
 
    @staticmethod
    def to_uint8(img: np.ndarray) -> np.ndarray:
        return np.clip(img * 255.0, 0, 255).astype(np.uint8)
 
    @staticmethod
    def gaussian_blur(img: np.ndarray, sigma: float) -> np.ndarray:
    
        return gaussian_filter(img.astype(np.float32), sigma=sigma)
 
    @staticmethod
    def laplacian_edge(img: np.ndarray, ksize: int = 3) -> np.ndarray:
        
        uint8 = ImageUtils.to_uint8(img)
        lap = cv2.Laplacian(uint8, cv2.CV_64F, ksize=ksize)
        return lap.astype(np.float32) / 255.0
 
    @staticmethod
    def apply_clahe(img: np.ndarray, clip: float = 2.0, tile: int = 8) -> np.ndarray:
        
        clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(tile, tile))
        return clahe.apply(img)
 
    @staticmethod
    def normalize(img: np.ndarray) -> np.ndarray:
        
        mn, mx = img.min(), img.max()
        if mx == mn:
            return np.zeros_like(img)
        return (img - mn) / (mx - mn)
 
    @staticmethod
    def compute_metrics(original: np.ndarray, enhanced: np.ndarray) -> dict:
        
        orig_u8 = original if original.dtype == np.uint8 else ImageUtils.to_uint8(original)
        enh_u8 = enhanced if enhanced.dtype == np.uint8 else ImageUtils.to_uint8(enhanced)
 
        psnr_val = psnr(orig_u8, enh_u8, data_range=255)
        ssim_val = ssim(orig_u8, enh_u8, data_range=255)
 
        # Edge strength via Sobel
        orig_edges = cv2.Sobel(orig_u8, cv2.CV_64F, 1, 0) ** 2 + cv2.Sobel(orig_u8, cv2.CV_64F, 0, 1) ** 2
        enh_edges  = cv2.Sobel(enh_u8,  cv2.CV_64F, 1, 0) ** 2 + cv2.Sobel(enh_u8,  cv2.CV_64F, 0, 1) ** 2
        edge_gain = np.sqrt(enh_edges.mean()) / (np.sqrt(orig_edges.mean()) + 1e-8)
 
        # Local contrast (std of local patches)
        local_std_orig = np.std(orig_u8.astype(float))
        local_std_enh  = np.std(enh_u8.astype(float))
        contrast_gain = local_std_enh / (local_std_orig + 1e-8)
 
        return {
            'PSNR (dB)': round(psnr_val, 2),
            'SSIM': round(ssim_val, 4),
            'Edge Strength Gain': round(edge_gain, 3),
            'Contrast Gain': round(contrast_gain, 3),
        }
 
 

In [ ]:
#  Pipeline 1: Unsharp Masking 
class UnsharpMaskingPipeline:
 
    def __init__(self, blur_sigma: float = 1.5, strength: float = 1.5,
                 clahe_clip: float = 1.5):
        self.blur_sigma = blur_sigma
        self.strength   = strength
        self.clahe_clip = clahe_clip
 
    def run(self, img_gray: np.ndarray) -> np.ndarray:
  
        f = ImageUtils.to_float(img_gray)
 
        # Step 1 – Gaussian blur
        blurred = ImageUtils.gaussian_blur(f, self.blur_sigma)
 
        # Step 2 – High-frequency mask
        mask = f - blurred
 
        # Step 3 – Add sharpened detail
        sharpened = f + self.strength * mask
 
        # Step 4 – CLAHE for global contrast
        sharp_u8 = ImageUtils.to_uint8(sharpened)
        result = ImageUtils.apply_clahe(sharp_u8, clip=self.clahe_clip, tile=8)
 
        return result
 
    @property
    def name(self):
        return "Pipeline 1\nUnsharp Masking"
 
    @property
    def description(self):
        return (
            "Unsharp Masking (USM)\n"
            f"σ={self.blur_sigma}, strength={self.strength}, CLAHE clip={self.clahe_clip}\n"
            "Subtracts blurred version to isolate high-freq detail,\n"
            "then adds weighted detail back to original."
        )
 
 

In [ ]:
 #  Pipeline 2: Laplacian Edge Boost + CLAHE
class LaplacianCLAHEPipeline:

 
    def __init__(self, clahe_clip1: float = 3.0, laplacian_weight: float = 0.6,
                 clahe_clip2: float = 1.5, ksize: int = 3):
        self.clahe_clip1       = clahe_clip1
        self.laplacian_weight  = laplacian_weight
        self.clahe_clip2       = clahe_clip2
        self.ksize             = ksize
 
    def run(self, img_gray: np.ndarray) -> np.ndarray:
        # Step 1 – Strong CLAHE
        equalized = ImageUtils.apply_clahe(img_gray, clip=self.clahe_clip1, tile=8)
 
        # Step 2 – Laplacian
        lap = cv2.Laplacian(equalized, cv2.CV_64F, ksize=self.ksize)
 
        # Step 3 – Subtract Laplacian (sharpening via second-derivative)
        sharpened = equalized.astype(np.float64) - self.laplacian_weight * lap
        sharpened = np.clip(sharpened, 0, 255).astype(np.uint8)
 
        # Step 4 – Final mild CLAHE
        result = ImageUtils.apply_clahe(sharpened, clip=self.clahe_clip2, tile=8)
        return result
 
    @property
    def name(self):
        return "Pipeline 2\nLaplacian + CLAHE"
 
    @property
    def description(self):
        return (
            "Laplacian Edge Boost + CLAHE\n"
            f"CLAHE clip={self.clahe_clip1}/{self.clahe_clip2}, "
            f"Lap weight={self.laplacian_weight}, ksize={self.ksize}\n"
            "Uses 2nd-order derivative to accentuate fine edges,\n"
            "sandwiched between two adaptive histogram passes."
        )
 
 

In [ ]:
#  Pipeline 3: Multi-Scale Retinex + HF Injection
 
class RetinexHFPipeline:

 
    def __init__(self, scales: list = None, hf_weight: float = 0.35,
                 clahe_clip: float = 2.0):
        self.scales     = scales or [15, 80, 250]   # small, medium, large σ
        self.hf_weight  = hf_weight
        self.clahe_clip = clahe_clip
 
    def _single_scale_retinex(self, img_f: np.ndarray, sigma: float) -> np.ndarray:
    
        blurred = ImageUtils.gaussian_blur(img_f + 1e-6, sigma)
        ssr = np.log1p(img_f) - np.log1p(blurred)
        return ssr
 
    def _multi_scale_retinex(self, img_f: np.ndarray) -> np.ndarray:
        
        msr = np.zeros_like(img_f)
        for s in self.scales:
            msr += self._single_scale_retinex(img_f, s)
        msr /= len(self.scales)
        return msr
 
    def _high_frequency_edges(self, img_u8: np.ndarray) -> np.ndarray:
    
        sx = cv2.Sobel(img_u8, cv2.CV_64F, 1, 0, ksize=3)
        sy = cv2.Sobel(img_u8, cv2.CV_64F, 0, 1, ksize=3)
        magnitude = np.sqrt(sx**2 + sy**2)
        return ImageUtils.normalize(magnitude)
 
    def run(self, img_gray: np.ndarray) -> np.ndarray:
        f = ImageUtils.to_float(img_gray)
 
        # Step 1 – Multi-Scale Retinex
        msr = self._multi_scale_retinex(f)
        msr_norm = ImageUtils.normalize(msr)
 
        # Step 2 – Inject HF edges
        hf = self._high_frequency_edges(img_gray)
        combined = msr_norm + self.hf_weight * hf
        combined = ImageUtils.normalize(combined)
 
        # Step 3 – CLAHE for clinical contrast
        combined_u8 = ImageUtils.to_uint8(combined)
        result = ImageUtils.apply_clahe(combined_u8, clip=self.clahe_clip, tile=8)
        return result
 
    @property
    def name(self):
        return "Pipeline 3\nRetinex + HF Inject"
 
    @property
    def description(self):
        return (
            "Multi-Scale Retinex + HF Injection\n"
            f"Scales={self.scales}, HF weight={self.hf_weight}, "
            f"CLAHE clip={self.clahe_clip}\n"
            "Decomposes illumination via log-domain multi-scale\n"
            "retinex, then reinjects Sobel edge detail."
        )
 

In [ ]:
#  XRayEnhancer – Orchestration Class

class XRayEnhancer:
 
    def __init__(self, image_path: str):
        raw = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        if raw is None:
            raise FileNotFoundError(f"Cannot read image: {image_path}")
        self.original = raw
        self.pipelines = [
            UnsharpMaskingPipeline(blur_sigma=1.5, strength=1.8, clahe_clip=1.8),
            LaplacianCLAHEPipeline(clahe_clip1=3.0, laplacian_weight=0.5, clahe_clip2=1.5),
            RetinexHFPipeline(scales=[15, 80, 250], hf_weight=0.4, clahe_clip=2.0),
        ]
        self.results = {}
        self.metrics = {}
 
    def run_all(self):
        for pipe in self.pipelines:
            enhanced = pipe.run(self.original)
            self.results[pipe.name] = enhanced
            self.metrics[pipe.name] = ImageUtils.compute_metrics(self.original, enhanced)
        return self
 
    def save_individual_outputs(self, prefix: str = "/home/claude/task2_pipeline"):
        for i, pipe in enumerate(self.pipelines, 1):
            out_path = f"{prefix}_{i}.png"
            cv2.imwrite(out_path, self.results[pipe.name])
        cv2.imwrite(f"{prefix}_original.png", self.original)
 
    def generate_comparison_figure(self, save_path: str = "/home/claude/task2_comparison.png"):
    
        fig = plt.figure(figsize=(22, 18), facecolor='#0d1117')
        fig.patch.set_facecolor('#0d1117')
 
        pipes = self.pipelines
        n = len(pipes)
 
        # ── Title ──────────────────────────────────
        fig.text(0.5, 0.97, 'X-Ray Image Enhancement — Comparative Sharpening Study',
                 ha='center', va='top', fontsize=18, fontweight='bold',
                 color='#e6edf3', fontfamily='DejaVu Sans')
        fig.text(0.5, 0.945,
                 'Task 2  |  Three independent enhancement pipelines applied to the same X-ray',
                 ha='center', va='top', fontsize=11, color='#8b949e')
 
        # ── Row 1: Images ──────────────────────────
        all_imgs = [self.original] + [self.results[p.name] for p in pipes]
        titles   = ['Original\n(Input)', pipes[0].name, pipes[1].name, pipes[2].name]
        colors   = ['#8b949e', '#58a6ff', '#3fb950', '#f78166']
 
        for col, (img, title, col_color) in enumerate(zip(all_imgs, titles, colors)):
            ax = fig.add_axes([0.03 + col * 0.245, 0.62, 0.215, 0.285])
            ax.imshow(img, cmap='gray', vmin=0, vmax=255)
            ax.set_title(title, color=col_color, fontsize=11, fontweight='bold', pad=8)
            ax.axis('off')
            for spine in ax.spines.values():
                spine.set_edgecolor(col_color)
                spine.set_linewidth(2)
 
        # ── Row 2: Histograms ──────────────────────
        for col, (img, title, col_color) in enumerate(zip(all_imgs, titles, colors)):
            ax = fig.add_axes([0.03 + col * 0.245, 0.40, 0.215, 0.175])
            ax.set_facecolor('#161b22')
            hist = cv2.calcHist([img], [0], None, [64], [0, 256]).flatten()
            ax.fill_between(range(64), hist, alpha=0.6, color=col_color)
            ax.plot(hist, color=col_color, linewidth=1.2)
            ax.set_xlim(0, 63)
            ax.set_xlabel('Intensity', color='#8b949e', fontsize=8)
            ax.set_ylabel('Count', color='#8b949e', fontsize=8)
            ax.tick_params(colors='#8b949e', labelsize=7)
            ax.set_title('Histogram', color='#8b949e', fontsize=9)
            for spine in ax.spines.values():
                spine.set_edgecolor('#30363d')
 
        # ── Row 3: Difference maps ─────────────────
        diff_label = ['', 'Diff: P1 − Original', 'Diff: P2 − Original', 'Diff: P3 − Original']
        for col in range(1, 4):
            pipe = pipes[col - 1]
            diff = cv2.absdiff(self.results[pipe.name], self.original)
            diff_eq = cv2.equalizeHist(diff)
            ax = fig.add_axes([0.03 + col * 0.245, 0.21, 0.215, 0.155])
            ax.imshow(diff_eq, cmap='inferno', vmin=0, vmax=255)
            ax.set_title(diff_label[col], color='#f0883e', fontsize=9, pad=5)
            ax.axis('off')
 
        ax0 = fig.add_axes([0.03, 0.21, 0.215, 0.155])
        ax0.set_facecolor('#161b22')
        ax0.text(0.5, 0.5, 'Difference Maps\n(|Enhanced − Original|)\nBrighter = More Change',
                 ha='center', va='center', color='#f0883e',
                 fontsize=10, transform=ax0.transAxes)
        ax0.axis('off')
 
        # ── Row 4: Metrics table ───────────────────
        metric_keys = ['PSNR (dB)', 'SSIM', 'Edge Strength Gain', 'Contrast Gain']
        col_labels = [p.name.replace('\n', ' ') for p in pipes]
        cell_data  = []
        for mk in metric_keys:
            row = [str(self.metrics[p.name][mk]) for p in pipes]
            cell_data.append(row)
 
        ax_table = fig.add_axes([0.03, 0.03, 0.94, 0.155])
        ax_table.set_facecolor('#161b22')
        ax_table.axis('off')
        ax_table.set_title('Quantitative Metrics Comparison', color='#e6edf3',
                           fontsize=12, fontweight='bold', pad=8, loc='left')
 
        tbl = ax_table.table(
            cellText=cell_data,
            rowLabels=metric_keys,
            colLabels=col_labels,
            cellLoc='center',
            loc='center',
            bbox=[0.12, 0.0, 0.86, 1.0]
        )
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(10)
        for (r, c), cell in tbl.get_celld().items():
            cell.set_facecolor('#161b22')
            cell.set_edgecolor('#30363d')
            cell.set_linewidth(0.8)
            if r == 0:
                cell.set_facecolor('#21262d')
                cell.set_text_props(color=colors[c] if c >= 0 else '#8b949e', fontweight='bold')
            elif c == -1:
                cell.set_text_props(color='#8b949e', fontweight='bold')
            else:
                cell.set_text_props(color='#e6edf3')
 
        plt.savefig(save_path, dpi=150, bbox_inches='tight',
                    facecolor='#0d1117', edgecolor='none')
        plt.close()
        print(f"Comparison figure saved → {save_path}")
        return save_path
 
    def print_analysis(self):
        """Print a structured text analysis to stdout."""
        print("\n" + "="*70)
        print("  TASK 2 — X-RAY SHARPENING PIPELINES: ANALYSIS REPORT")
        print("="*70)
 
        for pipe in self.pipelines:
            name = pipe.name.replace('\n', ' ')
            m = self.metrics[pipe.name]
            print(f"\n{'─'*60}")
            print(f"  {name}")
            print(f"{'─'*60}")
            print(pipe.description)
            print("\n  Metrics:")
            for k, v in m.items():
                print(f"    {k:<25}: {v}")
 
        print("\n" + "="*70)
        print("  COMPARATIVE SUMMARY")
        print("="*70)
        print("""
  Pipeline 1 – Unsharp Masking
    ✓ Best balance between sharpness and noise control.
    ✓ Predictable behavior; easy to tune clinically.
    ✗ Can produce mild halo artifacts at high strength.
 
  Pipeline 2 – Laplacian + CLAHE
    ✓ Highest edge accentuation — excellent for bone/vessel borders.
    ✓ CLAHE ensures adaptive local contrast enhancement.
    ✗ Second-order derivative amplifies noise; requires denoising pre-step.
 
  Pipeline 3 – Multi-Scale Retinex + HF Injection
    ✓ Best at correcting uneven exposure and scatter typical in X-rays.
    ✓ Preserves soft-tissue detail across bright and dark regions.
    ✗ Most computationally expensive; non-linear response can surprise.
        """)
        print("="*70)
 


In [13]:
if __name__ == "__main__":
    enhancer = XRayEnhancer("c:/Users/Delta/Pictures/Screenshots/OIP.webp")
    enhancer.run_all()
    enhancer.print_analysis()
    enhancer.save_individual_outputs()
    enhancer.generate_comparison_figure("c:/Users/Delta/Pictures/Screenshots/task2_comparisonimag4.png")
    print("\nAll outputs saved successfully.") 


  TASK 2 — X-RAY SHARPENING PIPELINES: ANALYSIS REPORT

────────────────────────────────────────────────────────────
  Pipeline 1 Unsharp Masking
────────────────────────────────────────────────────────────
Unsharp Masking (USM)
σ=1.5, strength=1.8, CLAHE clip=1.8
Subtracts blurred version to isolate high-freq detail,
then adds weighted detail back to original.

  Metrics:
    PSNR (dB)                : 22.24
    SSIM                     : 0.8443
    Edge Strength Gain       : 2.353
    Contrast Gain            : 1.102

────────────────────────────────────────────────────────────
  Pipeline 2 Laplacian + CLAHE
────────────────────────────────────────────────────────────
Laplacian Edge Boost + CLAHE
CLAHE clip=3.0/1.5, Lap weight=0.5, ksize=3
Uses 2nd-order derivative to accentuate fine edges,
sandwiched between two adaptive histogram passes.

  Metrics:
    PSNR (dB)                : 15.28
    SSIM                     : 0.4491
    Edge Strength Gain       : 3.538
    Contrast Gain    